# hotpotqa ở n=1500 - đồng đều cỡ mẫu với 2Wiki

Ở n=500 khẳng định trên hotpotqa: **cả ba khẳng định ĐÃ sạch ở n=500 (−7.99 chỉ cần n=128)**.

Kernel này **không** khép khẳng định nào - nó mua sự đồng đều. Sau khi
2Wiki đổi dấu từ n=500 sang n=1500, để một dataset ở n=500 trong khi hai cái kia
ở n≥1000 mời đúng câu hỏi *"chỉ chạy thêm ở chỗ có lợi à?"*. Chi phí 2.8h là rẻ
so với việc phải trả lời câu đó bằng lời.

| | |
|---|---|
| $k$ | 90 (đúng §5.1 của paper) |
| Toàn tập | 7,405 câu |
| Chi phí | 0.94h ở n=500 × 3 = **~2.8h** T4 |
| Ghi ra | `results/hotpotqa_1500_7b.json` - file n=500 giữ lại để đối chiếu |

Mọi cấu hình khác giữ y hệt lần n=500 (chỉ đổi `--limit`), nên hai lần chạy so
sánh được với nhau, và 500 câu đầu phải tái hiện tới 0.00 F1 vì decoding là
greedy - ô kiểm cuối kiểm đúng điều đó.


In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ hotpotqa, n=1500, reader 7B ══ ~2.8h
import subprocess, sys, os, time
READER='Qwen/Qwen2.5-7B-Instruct'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile):
    t0=time.time()
    with open(logfile,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}'); return p.returncode

OUT='results/hotpotqa_1500_7b.json'
if os.path.exists(OUT):
    print(f'[bo qua] da co {OUT}')
else:
    rc=run_stream([sys.executable,'-u','scripts/run_eval.py',
        '--dataset','hotpotqa','--limit','1500',
        '--reader','hf','--reader-model',READER,'--load-4bit',
        '--itercomp-llm','hf','--scorer','dual',
        '--methods','raw,oracle,llmlingua2,itercomp','--out',OUT],
        'results/log_hotpotqa_1500.txt')
    assert rc==0, f'hotpotqa n=1500 that bai (rc={rc})'


In [ ]:
import json, sys, os
sys.path.insert(0, 'repo/src' if os.path.isdir('repo/src') else 'src')
from itercomp.stats import paired_bootstrap
new=json.load(open('results/hotpotqa_1500_7b.json'))
old=json.load(open('results/hotpotqa_500_7b.json')) if os.path.exists('results/hotpotqa_500_7b.json') else None
G=lambda R,m:[100*r['methods'][m]['f1_norm'] for r in R]

def show(d,tag):
    R=d['per_row']; print(f'  {tag} (n={len(R)})')
    for a,b in (('itercomp','llmlingua2'),('itercomp','raw'),('oracle','raw')):
        r=paired_bootstrap(G(R,a),G(R,b))
        print(f"    {a:9s}-{b:11s} {r['diff']:+6.2f} [{r['lo']:+6.2f},{r['hi']:+6.2f}]  "
              f"{'SACH' if r['lo']*r['hi']>0 else 'cat qua 0'}")
if old: show(old,'n=500 (cu)')
show(new,'n=1500 (moi)')

if old:
    # 500 cau dau PHAI tai hien y het - greedy, cung bo cau, cung config.
    print()
    for m in ('oracle','raw','itercomp','llmlingua2'):
        a=sum(G(old['per_row'],m))/len(old['per_row'])
        b=sum(G(new['per_row'][:500],m))/500
        flag='OK' if abs(a-b)<0.01 else '*** LECH - hai lan chay KHONG so sanh duoc ***'
        print(f'  {m:11s} cu {a:6.2f}  moi(500 dau) {b:6.2f}  {flag}')
    o=paired_bootstrap(G(old['per_row'],'itercomp'),G(old['per_row'],'raw'))
    n=paired_bootstrap(G(new['per_row'],'itercomp'),G(new['per_row'],'raw'))
    print(f"\n  IterCOMP-Raw: n=500 {o['diff']:+.2f} -> n=1500 {n['diff']:+.2f}  "
          f"{'GIU DAU' if (o['diff']>0)==(n['diff']>0) else '*** DOI DAU - phai sua bao cao ***'}")
    print('  -> CI da sach' if n['lo']*n['hi']>0 else
          f"  -> van cat 0: hieu that su nho (±{max(abs(n['lo']),abs(n['hi'])):.1f})")


In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_hotpotqa_1500'),'zip','results')
print('✓',z,f'({os.path.getsize(z)/1e6:.1f} MB)')
